#### Bibliotecas

- Pandas : Tratar e cruzar os dados
- pyarrow : Salvar a Camada Ouro em parquet


In [1]:
!pip install pandas pyarrow fastparquet

In [2]:
import pandas as pd
import os
import unicodedata


In [3]:
# Pastas de dados
PASTA_DADOS = os.path.join("..", "dados")
PASTA_BRONZE = os.path.join(PASTA_DADOS, "bronze")
PASTA_GOLD = os.path.join(PASTA_DADOS, "gold")
os.makedirs(PASTA_GOLD, exist_ok=True)

In [4]:
# Funcoes e mapas de compatibilizacao de nomes de municipios
# As tres bases (IBGE, ANEEL e LABREN) grafam alguns municipios de forma
# diferente. Sem tratar isso, o merge falha silenciosamente e o municipio
# entra no dataset com zero conexoes - um dado falso, nao um dado ausente.
# A base de referencia para os nomes e o IBGE, usada na tabela de renda.

def normalizar(texto):
    """Remove acentos e padroniza caixa para comparar nomes entre bases."""
    texto = unicodedata.normalize("NFKD", str(texto))
    return texto.encode("ascii", "ignore").decode().upper().strip()


# Grafia da ANEEL -> grafia do IBGE
CORRECOES_ANEEL = {
    "ACU|RN":                     "ASSU|RN",
    "BOA SAUDE|RN":               "JANUARIO CICCO|RN",   # mesmo município
    "OLHO-D'AGUA DO BORGES|RN":   "OLHO D'AGUA DO BORGES|RN",
    "IGUARACI|PE":                "IGUARACY|PE",
    "LAGOA DO ITAENGA|PE":        "LAGOA DE ITAENGA|PE",
    "AMPARO DE SAO FRANCISCO|SE": "AMPARO DO SAO FRANCISCO|SE",
    "MUQUEM DE SAO FRANCISCO|BA": "MUQUEM DO SAO FRANCISCO|BA",
    "SANTA TERESINHA|BA":         "SANTA TEREZINHA|BA",
    "PINDARE MIRIM|MA":           "PINDARE-MIRIM|MA",
}

# Grafia do LABREN -> grafia do IBGE
CORRECOES_LABREN = {
    "ACU|RN":                     "ASSU|RN",
    "ARES|RN":                    "AREZ|RN",
    "AUGUSTO SEVERO|RN":          "CAMPO GRANDE|RN",     # renomeado em 2013
    "OLHO-D'AGUA DO BORGES|RN":   "OLHO D'AGUA DO BORGES|RN",
    "BELEM DE SAO FRANCISCO|PE":  "BELEM DO SAO FRANCISCO|PE",
    "LAGOA DO ITAENGA|PE":        "LAGOA DE ITAENGA|PE",
    "AMPARO DE SAO FRANCISCO|SE": "AMPARO DO SAO FRANCISCO|SE",
    "SANTA TERESINHA|BA":         "SANTA TEREZINHA|BA",
    "MUQUEM DE SAO FRANCISCO|BA": "MUQUEM DO SAO FRANCISCO|BA",
}

print(f"Correções carregadas — ANEEL: {len(CORRECOES_ANEEL)} | LABREN: {len(CORRECOES_LABREN)}")

Correções carregadas — ANEEL: 9 | LABREN: 9


In [5]:
# Le a Camada Bronze gerada pelo Extract.ipynb
df_renda_per_capita = pd.read_pickle(os.path.join(PASTA_BRONZE, "renda_per_capita.pkl"))
df_aneel = pd.read_pickle(os.path.join(PASTA_BRONZE, "aneel.pkl"))
df_pop = pd.read_pickle(os.path.join(PASTA_BRONZE, "populacao.pkl"))
df_area = pd.read_pickle(os.path.join(PASTA_BRONZE, "area.pkl"))
df_dens = pd.read_pickle(os.path.join(PASTA_BRONZE, "densidade.pkl"))
df_solar = pd.read_pickle(os.path.join(PASTA_BRONZE, "solar.pkl"))

In [6]:
# Camada Prata
# 1. Renda per capita - nivel MUNICIPAL, filtrado para o Nordeste

# Mapeamento UF -> Regiao (para filtrar o Nordeste)
REGIOES = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}
df_renda_per_capita = df_renda_per_capita.rename(columns={
    "Unnamed: 0": "municipio_uf",
    "2022": "renda_per_capita"
})
df_renda_per_capita["renda_per_capita"] = pd.to_numeric(
    df_renda_per_capita["renda_per_capita"], errors="coerce"
)
df_renda_per_capita = df_renda_per_capita.dropna(subset=["renda_per_capita"])

# Separa "Municipio (UF)" em duas colunas
df_renda_per_capita[["municipio", "uf"]] = (
    df_renda_per_capita["municipio_uf"].str.extract(r"^(.*)\s\((\w{2})\)$")
)
df_renda_per_capita["municipio"] = df_renda_per_capita["municipio"].str.strip()

# Regiao e ranking nacional (antes do filtro, para manter o contexto do pais)
df_renda_per_capita["regiao"] = df_renda_per_capita["uf"].map(REGIOES)
df_renda_per_capita["renda_per_capita"] = df_renda_per_capita["renda_per_capita"].round(2)
df_renda_per_capita = df_renda_per_capita.sort_values(
    "renda_per_capita", ascending=False
).reset_index(drop=True)
df_renda_per_capita["ranking_nacional"] = df_renda_per_capita.index + 1

assert len(df_renda_per_capita) == 5570, (
    f"Esperados 5570 municípios, encontrados {len(df_renda_per_capita)}"
)
assert df_renda_per_capita["regiao"].notna().all(), "UF sem região mapeada"
assert not df_renda_per_capita.duplicated(subset=["municipio", "uf"]).any(), (
    "Municípios duplicados"
)

# Filtra apenas o Nordeste
df_renda_silver = df_renda_per_capita[
    df_renda_per_capita["regiao"] == "Nordeste"
][["ranking_nacional", "municipio", "uf", "regiao", "renda_per_capita"]].copy()

print(f"Municípios do Nordeste: {len(df_renda_silver)}")
display(df_renda_silver)

Municípios do Nordeste: 1794


,ranking_nacional,municipio,uf,regiao,renda_per_capita
64,65,Fernando de Noronha,PE,Nordeste,2566.14
80,81,Eusébio,CE,Nordeste,2512.70
151,152,Recife,PE,Nordeste,2284.77
286,287,Aracaju,SE,Nordeste,2087.74
320,321,Natal,RN,Nordeste,2049.37
...,...,...,...,...,...
5562,5563,Humberto de Campos,MA,Nordeste,415.69
5563,5564,Primeira Cruz,MA,Nordeste,413.00
5565,5566,Cachoeira Grande,MA,Nordeste,389.45
5566,5567,Belágua,MA,Nordeste,388.46


In [7]:
# Camada Prata
# 2. ANEEL - solar residencial no Nordeste, agregado por MUNICIPIO
colunas_uteis = [
    'CodUFibge',
    'SigUF',
    'NomMunicipio',
    'DscFonteGeracao',
    'DscClasseConsumo',
    'MdaPotenciaInstaladaKW'
]

df_aneel_filtro = df_aneel[colunas_uteis].copy()

df_aneel_filtro['MdaPotenciaInstaladaKW'] = (
    df_aneel_filtro['MdaPotenciaInstaladaKW']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

# Filtrar Solar Residencial
df_solar_aneel = df_aneel_filtro[
    (df_aneel_filtro['DscFonteGeracao'].str.contains('Radiação solar|Solar', case=False, na=False)) &
    (df_aneel_filtro['DscClasseConsumo'].str.contains('Residencial', case=False, na=False))
].copy()

# Filtrar Nordeste
codigos_nordeste = [21, 22, 23, 24, 25, 26, 27, 28, 29]
df_solar_ne = df_solar_aneel[
    df_solar_aneel['CodUFibge'].isin(codigos_nordeste)
].copy()

# Agregacao por municipio
df_aneel_silver = df_solar_ne.groupby(['SigUF', 'NomMunicipio']).agg(
    Qtd_Conexoes_Total=('MdaPotenciaInstaladaKW', 'count'),
    Potencia_Instalada_KW_Total=('MdaPotenciaInstaladaKW', 'sum')
).reset_index()

display(df_aneel_silver)

,SigUF,NomMunicipio,Qtd_Conexoes_Total,Potencia_Instalada_KW_Total
0,AL,Anadia,173,1368.01
1,AL,Arapiraca,5704,40294.18
2,AL,Atalaia,247,2048.50
3,AL,Barra de Santo Antônio,311,2525.13
4,AL,Barra de São Miguel,745,7150.25
...,...,...,...,...
1789,SE,São Miguel do Aleixo,12,144.10
1790,SE,Telha,12,70.46
1791,SE,Tobias Barreto,302,2147.22
1792,SE,Tomar do Geru,26,116.44


In [8]:
# Camada Prata
# 3. Populacao, area e densidade demografica - nivel Municipal (atualizado)
def tratar_tabela(df, nome_coluna):
    df = df.iloc[3:].copy()
    df.columns = ["municipio_uf", nome_coluna]
    df = df.dropna()
    df["municipio_uf"] = df["municipio_uf"].astype(str).str.strip()

    # Separa "Municipio (UF)" em duas colunas, igual ao tratamento da renda per capita
    df[["municipio", "uf"]] = df["municipio_uf"].str.extract(r"^(.*)\s\((\w{2})\)$")
    df["municipio"] = df["municipio"].str.strip()
    # Usa normalizar() - a MESMA regra das chaves de renda, ANEEL e LABREN.
    # Com .str.upper() os acentos eram preservados e "SÃO JOSÉ" nao casava
    # com "SAO JOSE", fazendo 769 municípios perderem população e densidade
    # silenciosamente. Toda chave do projeto passa por normalizar().
    df["_chave"] = df["municipio"].map(normalizar) + "|" + df["uf"]

    return df[["_chave", nome_coluna]]

# Usa variaveis novas (df_*_tratado) em vez de sobrescrever df_pop/df_area/df_dens:
# assim a celula fica segura para ser executada mais de uma vez no mesmo kernel
# (reexecutar sem reler o pickle nao quebra mais o tratamento)
df_pop_tratado = tratar_tabela(df_pop, "populacao_residente")
df_area_tratado = tratar_tabela(df_area, "area_unidade_territorial_km2")
df_dens_tratado = tratar_tabela(df_dens, "densidade_demografica")

df_pop_tratado["populacao_residente"] = pd.to_numeric(df_pop_tratado["populacao_residente"], errors="coerce")
df_area_tratado["area_unidade_territorial_km2"] = pd.to_numeric(df_area_tratado["area_unidade_territorial_km2"], errors="coerce")
df_dens_tratado["densidade_demografica"] = pd.to_numeric(df_dens_tratado["densidade_demografica"], errors="coerce")

df_area_densidade = (
    df_pop_tratado
    .merge(df_area_tratado, on="_chave", how="left")
    .merge(df_dens_tratado, on="_chave", how="left")
)

display(df_area_densidade.head())

,_chave,populacao_residente,area_unidade_territorial_km2,densidade_demografica
0,ALTA FLORESTA D'OESTE|RO,21494,7067.127,3.04
1,ARIQUEMES|RO,96833,4426.571,21.88
2,CABIXI|RO,5351,1314.352,4.07
3,CACOAL|RO,86887,3793.000,22.91
4,CEREJEIRAS|RO,15890,2783.300,5.71


In [9]:
# Camada Prata
# 4. Irradiacao solar (GHI) - nivel MUNICIPAL (LABREN/INPE)
# Substitui o antigo dicionario ghi_por_estado: cada municipio recebe o GHI
# medido na sua propria sede, e nao mais a media do estado inteiro.
UF_LABREN = {
    "MARANHÃO": "MA", "PIAUÍ": "PI", "CEARÁ": "CE", "RIO GRANDE DO NORTE": "RN",
    "PARAÍBA": "PB", "PERNAMBUCO": "PE", "ALAGOAS": "AL", "SERGIPE": "SE",
    "BAHIA": "BA",
}

df_solar_silver = df_solar.copy()
df_solar_silver["uf"] = df_solar_silver["STATE"].map(UF_LABREN)
df_solar_silver = df_solar_silver.dropna(subset=["uf"])          # so o Nordeste

# Wh/m2/dia (origem) -> kWh/m2/dia (unidade usada no TCC)
df_solar_silver["ghi_media_kwh_m2_dia"] = (
    df_solar_silver["ANNUAL"] / 1000
).round(3)

df_solar_silver["_chave"] = (
    df_solar_silver["NAME"].map(normalizar) + "|" + df_solar_silver["uf"]
).replace(CORRECOES_LABREN)

df_solar_silver = df_solar_silver[["_chave", "ghi_media_kwh_m2_dia"]]

assert not df_solar_silver["_chave"].duplicated().any(), "chave de GHI duplicada"

print(f"Municípios do Nordeste com GHI: {len(df_solar_silver)}")
display(df_solar_silver.head())

Municípios do Nordeste com GHI: 1794


,_chave,ghi_media_kwh_m2_dia
22,PIACABUCU|AL,5.441
23,PENEDO|AL,5.338
24,FELIZ DESERTO|AL,5.439
25,PORTO REAL DO COLEGIO|AL,5.434
26,SAO BRAS|AL,5.399


In [10]:
# Camada Ouro - nivel MUNICIPAL
# Passo 1 - Renda (municipal) + ANEEL (municipal), casando por municipio + uf
# As chaves usam normalizar() (sem acento) para que diferencas de acentuacao
# entre as bases nao quebrem o merge, e CORRECOES_ANEEL para as grafias que
# divergem por completo. Sem isso, 14 municipios entravam com zero conexoes -
# um dado falso que distorce tanto o modelo quanto o ranking final.
df_renda_silver["_chave"] = (
    df_renda_silver["municipio"].map(normalizar) + "|" + df_renda_silver["uf"]
)
df_aneel_silver["_chave"] = (
    df_aneel_silver["NomMunicipio"].map(normalizar) + "|" + df_aneel_silver["SigUF"]
).replace(CORRECOES_ANEEL)

# Se a compatibilizacao estiver correta, todo municipio da ANEEL encontra par
nao_casados = set(df_aneel_silver["_chave"]) - set(df_renda_silver["_chave"])
assert not nao_casados, f"Municípios da ANEEL sem par no IBGE: {sorted(nao_casados)}"

df_dataset = df_renda_silver.merge(
    df_aneel_silver.drop(columns=["SigUF", "NomMunicipio"]),
    on="_chave",
    how="left"
)

# Apos a compatibilizacao de nomes acima, um municipio ausente da ANEEL
# significa de fato nenhuma conexao solar residencial cadastrada - e nao
# uma falha de merge. So por isso o fillna(0) e legitimo aqui.
df_dataset["Qtd_Conexoes_Total"] = df_dataset["Qtd_Conexoes_Total"].fillna(0)
df_dataset["Potencia_Instalada_KW_Total"] = pd.to_numeric(
    df_dataset["Potencia_Instalada_KW_Total"], errors="coerce"
).fillna(0)

# Passo 2 - Adicionar Area, Populacao e Densidade do proprio MUNICIPIO
# (a planilha do IBGE passou a trazer os dados por municipio; casamos pela mesma
# chave municipio+uf do Passo 1, em vez de replicar o valor do estado inteiro)
df_dataset = df_dataset.merge(
    df_area_densidade,
    on="_chave",
    how="left"
).drop(columns=["_chave"])

# Passo 3 - Nome completo do estado (para leitura das tabelas e graficos)
UF_PARA_ESTADO = {
    "MA": "Maranhão", "PI": "Piauí", "CE": "Ceará", "RN": "Rio Grande do Norte",
    "PB": "Paraíba", "PE": "Pernambuco", "AL": "Alagoas", "SE": "Sergipe", "BA": "Bahia",
}
df_dataset["Estado"] = df_dataset["uf"].map(UF_PARA_ESTADO)

# Passo 4 - GHI por MUNICIPIO (LABREN/INPE)
# Antes: um unico valor por estado, replicado para todos os seus municipios.
# Agora: o GHI da sede de cada municipio, o que devolve variacao real dentro
# do estado (ex.: na Bahia o GHI vai de 4,694 a 6,021 kWh/m2/dia).
df_dataset["_chave_ghi"] = (
    df_dataset["municipio"].map(normalizar) + "|" + df_dataset["uf"]
)
df_dataset = df_dataset.merge(
    df_solar_silver.rename(columns={"_chave": "_chave_ghi"}),
    on="_chave_ghi",
    how="left"
).drop(columns=["_chave_ghi"])

# Fernando de Noronha (PE) fica sem GHI: o arquivo do LABREN cobre apenas o
# territorio continental, e o arquipelago esta a ~350 km da costa.
sem_ghi = df_dataset[df_dataset["ghi_media_kwh_m2_dia"].isna()]
if len(sem_ghi):
    print(f"Municipios sem GHI (fora do grid continental do LABREN): "
          f"{sem_ghi['municipio'].tolist()}")

print(f"Shape do dataset final (municipal): {df_dataset.shape}")
display(df_dataset)

# Trava de integridade: nenhuma variavel do modelo pode sair da Camada Ouro
# com valor nulo. Um merge que falha por divergencia de acentuacao ou grafia
# gera nulos em massa sem levantar erro - foi o que ocorreu quando as chaves
# de populacao e de renda usavam regras de normalizacao diferentes, e 769
# municipios perderam populacao sem nenhum aviso. O assert abaixo faz o
# notebook parar aqui, em vez de exportar um dataset pela metade.
COLUNAS_MODELO = [
    "renda_per_capita", "populacao_residente",
    "densidade_demografica", "Qtd_Conexoes_Total",
]
nulos = df_dataset[COLUNAS_MODELO].isnull().sum()
assert nulos.sum() == 0, (
    f"Merge incompleto — colunas com valores nulos:\n{nulos[nulos > 0]}"
)

# O GHI e a unica excecao tolerada: Fernando de Noronha esta fora do grid
# continental do LABREN. Qualquer numero acima de 1 indica problema no join.
sem_ghi = df_dataset["ghi_media_kwh_m2_dia"].isnull().sum()
assert sem_ghi <= 1, f"Esperado no máximo 1 município sem GHI, encontrados {sem_ghi}"

print(f"Integridade OK — {len(df_dataset)} municípios, nenhuma variável nula ✓")

Municipios sem GHI (fora do grid continental do LABREN): ['Fernando de Noronha']
Shape do dataset final (municipal): (1794, 12)


,ranking_nacional,municipio,uf,regiao,renda_per_capita,Qtd_Conexoes_Total,Potencia_Instalada_KW_Total,populacao_residente,area_unidade_territorial_km2,densidade_demografica,Estado,ghi_media_kwh_m2_dia
0,65,Fernando de Noronha,PE,Nordeste,2566.14,25,386.07,3167,18.609,170.19,Pernambuco,NaN
1,81,Eusébio,CE,Nordeste,2512.70,5370,42413.98,74170,78.818,941.03,Ceará,5.730
2,152,Recife,PE,Nordeste,2284.77,11851,82064.10,1488920,218.843,6803.60,Pernambuco,5.462
3,287,Aracaju,SE,Nordeste,2087.74,8086,62395.61,602757,182.163,3308.89,Sergipe,5.496
4,321,Natal,RN,Nordeste,2049.37,18634,124543.84,751300,167.401,4488.03,Rio Grande do Norte,5.674
...,...,...,...,...,...,...,...,...,...,...,...,...
1789,5563,Humberto de Campos,MA,Nordeste,415.69,39,750.30,25680,1714.625,14.98,Maranhão,5.253
1790,5564,Primeira Cruz,MA,Nordeste,413.00,22,217.59,13614,1337.161,10.18,Maranhão,5.304
1791,5566,Cachoeira Grande,MA,Nordeste,389.45,5,43.00,9732,707.236,13.76,Maranhão,4.986
1792,5567,Belágua,MA,Nordeste,388.46,6,77.95,8460,569.606,14.85,Maranhão,5.337


Integridade OK — 1794 municípios, nenhuma variável nula ✓


In [11]:
print("=== CHECAGEM DE QUALIDADE ===")
print(f"\nLinhas: {len(df_dataset)}")
print(f"Colunas: {list(df_dataset.columns)}")
print(f"\nValores nulos por coluna:")
print(df_dataset.isnull().sum())

print(f"\nMunicípios com pelo menos 1 conexão solar residencial: {(df_dataset['Qtd_Conexoes_Total'] > 0).sum()}")

# Zeros remanescentes: devem ser poucos e em municipios pequenos. Um municipio
# grande com zero conexoes e sinal de merge malsucedido, nao de ausencia real.
zeros = df_dataset[df_dataset["Qtd_Conexoes_Total"] == 0]
print(f"\n=== MUNICÍPIOS COM ZERO CONEXÕES: {len(zeros)} ===")
if len(zeros):
    print(zeros.nlargest(10, "populacao_residente")[
        ["municipio", "uf", "populacao_residente"]
    ].to_string(index=False))

# GHI agora varia DENTRO de cada estado - antes era constante por estado
print("\n=== GHI MUNICIPAL POR ESTADO (kWh/m²/dia) ===")
print(
    df_dataset.groupby("Estado")["ghi_media_kwh_m2_dia"]
    .agg(["count", "min", "mean", "max"])
    .round(3)
    .sort_values("mean", ascending=False)
)

=== CHECAGEM DE QUALIDADE ===

Linhas: 1794
Colunas: ['ranking_nacional', 'municipio', 'uf', 'regiao', 'renda_per_capita', 'Qtd_Conexoes_Total', 'Potencia_Instalada_KW_Total', 'populacao_residente', 'area_unidade_territorial_km2', 'densidade_demografica', 'Estado', 'ghi_media_kwh_m2_dia']

Valores nulos por coluna:
ranking_nacional                0
municipio                       0
uf                              0
regiao                          0
renda_per_capita                0
Qtd_Conexoes_Total              0
Potencia_Instalada_KW_Total     0
populacao_residente             0
area_unidade_territorial_km2    0
densidade_demografica           0
Estado                          0
ghi_media_kwh_m2_dia            1
dtype: int64

Municípios com pelo menos 1 conexão solar residencial: 1794

=== MUNICÍPIOS COM ZERO CONEXÕES: 0 ===

=== GHI MUNICIPAL POR ESTADO (kWh/m²/dia) ===
                     count    min   mean    max
Estado                                         
Rio Grande do Nor

In [12]:
# Persiste a Camada Ouro em disco para a proxima etapa (Load.ipynb) ler
df_dataset.to_parquet(os.path.join(PASTA_GOLD, "dataset_energia_solar.parquet"), index=False)
print(f"Camada Ouro salva em: {os.path.abspath(PASTA_GOLD)}")

Camada Ouro salva em: C:\Users\alvar\OneDrive\Engenharia de Software\Trabalho de Conclusão de Curso\Pipeline de Dados\Pipe\dados\gold
